### 03: EDA по таблице main_data_with_weather (погоды)

In [ ]:
df = df_main_with_weather.copy()

tech_cols = ['Cell Avail 2G (%)', 'Cell Avail 3G (%)', 'Cell Avail 4G (%)', 'Cell Avail Base Tech (%)']
weather_cols = ['TempDelta', 'MaxWind', 'MaxInt', 'Rain', 'Snow', 'Sprinkling', 'Hail', 'U', 'Po']

tech_cols = [col for col in tech_cols if col in df.columns]
weather_cols = [col for col in weather_cols if col in df.columns]

- Корреляция (все данные)
- Корреляция (доступность <99.8%)
- Влияние погоды на Cell Avail Base Tech (%) (все данные)
- Влияние погоды на Cell Avail Base Tech (%) (доступность <99.8%)  
Сохранено в `correlation_analysis.png`
- Корреляция между сгенерированными признаками
- Корреляция сгенерированных признаков с таргетом
####

In [ ]:
df_analysis = df[tech_cols + weather_cols].copy()
for col in df_analysis.columns:
    df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')
df_analysis.dropna(inplace=True)

corr_matrix = df_analysis.corr()

df_problems = df_analysis[df_analysis['Cell Avail Base Tech (%)'] < 99.8].copy()
corr_problems = df_problems[tech_cols + weather_cols].corr() if len(df_problems) > 0 else None

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

if len(tech_cols) > 0 and len(weather_cols) > 0:
    corr_subset = corr_matrix.loc[tech_cols, weather_cols]
    sns.heatmap(corr_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=axes[0, 0],
                annot_kws={'size': 12})
    axes[0, 0].set_title('Корреляция: Все данные', fontsize=16, fontweight='bold')
    axes[0, 0].tick_params(labelsize=12)
    plt.setp(axes[0, 0].get_xticklabels(), rotation=45, ha='right', fontsize=12)
    plt.setp(axes[0, 0].get_yticklabels(), fontsize=12)

if corr_problems is not None and len(df_problems) > 0:
    corr_problems_subset = corr_problems.loc[tech_cols, weather_cols]
    sns.heatmap(corr_problems_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=axes[0, 1],
                annot_kws={'size': 12})
    axes[0, 1].set_title(f'Корреляция: Доступность < 99.8% (n={len(df_problems)})', fontsize=16, fontweight='bold')
    axes[0, 1].tick_params(labelsize=12)
    plt.setp(axes[0, 1].get_xticklabels(), rotation=45, ha='right', fontsize=12)
    plt.setp(axes[0, 1].get_yticklabels(), fontsize=12)

fail_col = 'Cell Avail Base Tech (%)' if 'Cell Avail Base Tech (%)' in tech_cols else tech_cols[0] if tech_cols else None
if fail_col and len(weather_cols) > 0:
    fail_corrs = corr_matrix[fail_col][weather_cols].sort_values(ascending=False)
    colors = ['#0F2D69' if x > 0 else '#0FA0D7' for x in fail_corrs.values]  # R15 G45 B105 и R15 G160 B215
    axes[1, 0].barh(range(len(fail_corrs)), fail_corrs.values, color=colors, alpha=0.8)
    axes[1, 0].set_yticks(range(len(fail_corrs)))
    axes[1, 0].set_yticklabels(fail_corrs.index, fontsize=12)
    axes[1, 0].axvline(x=0, color='black', linewidth=0.8)
    axes[1, 0].set_xlabel('Коэффициент корреляции', fontsize=14)
    axes[1, 0].set_title(f'Влияние погоды на {fail_col} (все данные)', fontsize=16, fontweight='bold')
    axes[1, 0].set_xlim([-1, 1])
    axes[1, 0].tick_params(labelsize=12)
    for i, v in enumerate(fail_corrs.values):
        axes[1, 0].text(v + (0.05 if v > 0 else -0.15), i, f'{v:.3f}', va='center', fontsize=11)

if fail_col and len(weather_cols) > 0 and corr_problems is not None:
    fail_corrs_problems = corr_problems[fail_col][weather_cols].sort_values(ascending=False)
    colors = ['#0F2D69' if x > 0 else '#0FA0D7' for x in fail_corrs_problems.values]  # R15 G45 B105 и R15 G160 B215
    axes[1, 1].barh(range(len(fail_corrs_problems)), fail_corrs_problems.values, color=colors, alpha=0.8)
    axes[1, 1].set_yticks(range(len(fail_corrs_problems)))
    axes[1, 1].set_yticklabels(fail_corrs_problems.index, fontsize=12)
    axes[1, 1].axvline(x=0, color='black', linewidth=0.8)
    axes[1, 1].set_xlabel('Коэффициент корреляции', fontsize=14)
    axes[1, 1].set_title(f'Влияние погоды на {fail_col} (< 99.8%)', fontsize=16, fontweight='bold')
    axes[1, 1].set_xlim([-1, 1])
    axes[1, 1].tick_params(labelsize=12)
    for i, v in enumerate(fail_corrs_problems.values):
        axes[1, 1].text(v + (0.05 if v > 0 else -0.15), i, f'{v:.3f}', va='center', fontsize=11)

plt.suptitle('Анализ корреляции', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nВСЕГО ЗАПИСЕЙ: {len(df_analysis)}")
print(f"ЗАПИСЕЙ С ДОСТУПНОСТЬЮ < 99.8%: {len(df_problems)}\n")

if fail_col and len(weather_cols) > 0:
    print(f"КОРРЕЛЯЦИИ С {fail_col} (ВСЕ ДАННЫЕ):")
    print("-" * 50)
    for weather_col in weather_cols:
        corr = df_analysis[fail_col].corr(df_analysis[weather_col])
        p_value = stats.pearsonr(df_analysis[fail_col].dropna(), df_analysis[weather_col].dropna())[1]
        print(f"{weather_col:15s}: r = {corr:7.3f} (p = {p_value:.4f})")

if corr_problems is not None and len(df_problems) > 0:
    print(f"\nКОРРЕЛЯЦИИ С {fail_col} (ДОСТУПНОСТЬ < 99.8%):")
    print("-" * 50)
    for weather_col in weather_cols:
        corr = corr_problems[fail_col][weather_col]
        if len(df_problems[weather_col].dropna()) > 2:
            p_value = stats.pearsonr(df_problems[fail_col].dropna(), df_problems[weather_col].dropna())[1]
            print(f"{weather_col:15s}: r = {corr:7.3f} (p = {p_value:.4f})")

- Корреляция без сдвига (все данные)
- Корреляция (Погода со сдвигом +1 день)
- Влияние погоды на Cell Avail Base Tech (%) (без сдвига)
- Влияние погоды на Cell Avail Base Tech (%) (сдвиг +1 день)  
Сохранено в `correlation_with_shift.png`
####

In [ ]:
if 'Date' in df.columns or 'Дата' in df.columns:
    date_col = 'Date' if 'Date' in df.columns else 'Дата'
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

    df_analysis = df[tech_cols + weather_cols].copy()
    for col in df_analysis.columns:
        df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')
    df_analysis.dropna(inplace=True)

    df_shifted = df[tech_cols + weather_cols + [date_col]].copy()
    for col in weather_cols:
        df_shifted[col] = df_shifted[col].shift(1)
    for col in df_shifted.columns:
        if col != date_col:
            df_shifted[col] = pd.to_numeric(df_shifted[col], errors='coerce')
    df_shifted.dropna(inplace=True)

    corr_matrix = df_analysis.corr()
    corr_shifted = df_shifted[tech_cols + weather_cols].corr() if len(df_shifted) > 0 else None
else:
    df_analysis = df[tech_cols + weather_cols].copy()
    for col in df_analysis.columns:
        df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')
    df_analysis.dropna(inplace=True)
    corr_matrix = df_analysis.corr()
    corr_shifted = None
    print("ВНИМАНИЕ: Колонка с датой не найдена. Анализ со сдвигом невозможен.")

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

if len(tech_cols) > 0 and len(weather_cols) > 0:
    corr_subset = corr_matrix.loc[tech_cols, weather_cols]
    sns.heatmap(corr_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=axes[0, 0],
                annot_kws={'size': 11})
    axes[0, 0].set_title('Корреляция: Все данные (без сдвига)', fontsize=14, fontweight='bold')
    axes[0, 0].tick_params(labelsize=10)
    plt.setp(axes[0, 0].get_xticklabels(), rotation=45, ha='right', fontsize=10)
    plt.setp(axes[0, 0].get_yticklabels(), fontsize=10)

if corr_shifted is not None and len(df_shifted) > 0:
    corr_shifted_subset = corr_shifted.loc[tech_cols, weather_cols]
    sns.heatmap(corr_shifted_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=axes[0, 1],
                annot_kws={'size': 11})
    axes[0, 1].set_title(f'Корреляция: Погода со сдвигом +1 день (n={len(df_shifted)})', fontsize=14, fontweight='bold')
    axes[0, 1].tick_params(labelsize=10)
    plt.setp(axes[0, 1].get_xticklabels(), rotation=45, ha='right', fontsize=10)
    plt.setp(axes[0, 1].get_yticklabels(), fontsize=10)

fail_col = 'Cell Avail Base Tech (%)' if 'Cell Avail Base Tech (%)' in tech_cols else tech_cols[0] if tech_cols else None
if fail_col and len(weather_cols) > 0:
    fail_corrs = corr_matrix[fail_col][weather_cols].sort_values(ascending=False)
    colors = ['#0F2D69' if x > 0 else '#0FA0D7' for x in fail_corrs.values]
    axes[1, 0].barh(range(len(fail_corrs)), fail_corrs.values, color=colors, alpha=0.8)
    axes[1, 0].set_yticks(range(len(fail_corrs)))
    axes[1, 0].set_yticklabels(fail_corrs.index, fontsize=11)
    axes[1, 0].axvline(x=0, color='black', linewidth=0.8)
    axes[1, 0].set_xlabel('Коэффициент корреляции', fontsize=12)
    axes[1, 0].set_title(f'Влияние погоды на {fail_col} (без сдвига)', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlim([-1, 1])
    axes[1, 0].tick_params(labelsize=10)
    for i, v in enumerate(fail_corrs.values):
        axes[1, 0].text(v + (0.05 if v > 0 else -0.15), i, f'{v:.3f}', va='center', fontsize=10)

if fail_col and len(weather_cols) > 0 and corr_shifted is not None:
    fail_corrs_shifted = corr_shifted[fail_col][weather_cols].sort_values(ascending=False)
    colors = ['#0F2D69' if x > 0 else '#0FA0D7' for x in fail_corrs_shifted.values]
    axes[1, 1].barh(range(len(fail_corrs_shifted)), fail_corrs_shifted.values, color=colors, alpha=0.8)
    axes[1, 1].set_yticks(range(len(fail_corrs_shifted)))
    axes[1, 1].set_yticklabels(fail_corrs_shifted.index, fontsize=11)
    axes[1, 1].axvline(x=0, color='black', linewidth=0.8)
    axes[1, 1].set_xlabel('Коэффициент корреляции', fontsize=12)
    axes[1, 1].set_title(f'Влияние погоды на {fail_col} (сдвиг +1 день)', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlim([-1, 1])
    axes[1, 1].tick_params(labelsize=10)
    for i, v in enumerate(fail_corrs_shifted.values):
        axes[1, 1].text(v + (0.05 if v > 0 else -0.15), i, f'{v:.3f}', va='center', fontsize=10)

plt.suptitle('Анализ корреляции со сдвигом по времени', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_with_shift.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nВСЕГО ЗАПИСЕЙ: {len(df_analysis)}")
if corr_shifted is not None:
    print(f"ЗАПИСЕЙ ДЛЯ АНАЛИЗА СО СДВИГОМ: {len(df_shifted)}\n")

if fail_col and len(weather_cols) > 0:
    print(f"КОРРЕЛЯЦИИ С {fail_col} (БЕЗ СДВИГА):")
    print("-" * 50)
    for weather_col in weather_cols:
        corr = df_analysis[fail_col].corr(df_analysis[weather_col])
        p_value = stats.pearsonr(df_analysis[fail_col].dropna(), df_analysis[weather_col].dropna())[1]
        print(f"{weather_col:15s}: r = {corr:7.3f} (p = {p_value:.4f})")

if corr_shifted is not None and len(df_shifted) > 0:
    print(f"\nКОРРЕЛЯЦИИ С {fail_col} (ПОГОДА СО СДВИГОМ +1 ДЕНЬ):")
    print("-" * 50)
    for weather_col in weather_cols:
        corr = corr_shifted[fail_col][weather_col]
        if len(df_shifted[weather_col].dropna()) > 2:
            p_value = stats.pearsonr(df_shifted[fail_col].dropna(), df_shifted[weather_col].dropna())[1]
            print(f"{weather_col:15s}: r = {corr:7.3f} (p = {p_value:.4f})")

    print(f"\nСРАВНЕНИЕ КОРРЕЛЯЦИЙ:")
    print("-" * 50)
    for weather_col in weather_cols:
        corr_no_shift = df_analysis[fail_col].corr(df_analysis[weather_col])
        corr_with_shift = corr_shifted[fail_col][weather_col]
        diff = abs(corr_with_shift) - abs(corr_no_shift)
        marker = "↑" if diff > 0.05 else ("↓" if diff < -0.05 else "=")
        print(f"{weather_col:15s}: без сдвига={corr_no_shift:6.3f}, со сдвигом={corr_with_shift:6.3f} {marker}")

- Корреляция < 99.8% (сдвиг погоды внутри вышки)  
Сохранено в `correlation_per_tower_shift.png`
- Динамика корреляции Cell Avail Base Tech (%) (сдвиг внутри вышки)  
Сохранено в `correlation_dynamics_per_tower.png`
- Таблица корреляций для Cell Avail Base Tech (%)
####

In [ ]:
tower_col = None
for col in df.columns:
    if any(x in col.lower() for x in ['tower', 'cell', 'site', 'id', 'вышка', 'базовая']):
        if col not in tech_cols + weather_cols:
            tower_col = col
            break

date_col = None
if 'Date' in df.columns: date_col = 'Date'
elif 'Дата' in df.columns: date_col = 'Дата'

if not tower_col or not date_col:
    print(f"Не найдены нужные колонки. Найдено: tower={tower_col}, date={date_col}")
    print(f"Доступные колонки: {list(df.columns)}")
    exit()

print(f"Используем: вышка='{tower_col}', дата='{date_col}'")

df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values([tower_col, date_col]).reset_index(drop=True)

df_filtered = df[df['Cell Avail Base Tech (%)'] < 99.8].copy()
print(f"Записей с доступностью < 99.8%: {len(df_filtered)}")
print(f"Уникальных вышек: {df_filtered[tower_col].nunique()}")

def get_shifted_weather(df, weather_cols, tower_col, date_col, shift_days):
    """Сдвигает погодные колонки на shift_days дней ВНУТРИ каждой вышки"""
    result = df.copy()
    for col in weather_cols:
        result[col] = df.groupby(tower_col)[col].shift(shift_days)
    return result

shifts = [0, 1, 2, 3, 4, 5, 6, 7]
correlations = {}

for s in shifts:
    if s == 0:
        data = df_filtered[tech_cols + weather_cols].dropna()
    else:
        data = df_filtered[tech_cols + weather_cols + [tower_col, date_col]].copy()
        for col in weather_cols:
            data[col] = df_filtered.groupby(tower_col)[col].shift(s)
        data = data.dropna()

    if len(data) > 10:
        key = 'No Shift' if s == 0 else f'Shift +{s} Day' if s == 1 else f'Shift +{s} Days'
        correlations[key] = data[tech_cols + weather_cols].corr()
        print(f"{key}: {len(data)} записей")

In [ ]:
fail_col = 'Cell Avail Base Tech (%)'

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
plot_labels = [k for k in correlations.keys() if 'No Shift' in k or '+1 Day' in k or '+2 Day' in k or '+7 Days' in k][:4]

for i, label in enumerate(plot_labels):
    corr = correlations[label]

    sns.heatmap(corr.loc[tech_cols, weather_cols], annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, linewidths=0.3, ax=axes[0, i], cbar=False)
    axes[0, i].set_title(label, fontsize=10, fontweight='bold')
    plt.setp(axes[0, i].get_xticklabels(), rotation=45, ha='right', fontsize=8)

    if fail_col in tech_cols:
        fail_corrs = corr[fail_col][weather_cols].sort_values(ascending=False)
        colors = ['red' if x > 0 else 'blue' for x in fail_corrs.values]
        axes[1, i].barh(range(len(fail_corrs)), fail_corrs.values, color=colors, alpha=0.7)
        axes[1, i].set_yticks(range(len(fail_corrs)))
        axes[1, i].set_yticklabels(fail_corrs.index, fontsize=8)
        axes[1, i].axvline(x=0, color='black', linewidth=0.5)
        axes[1, i].set_xlim([-1, 1])
        axes[1, i].set_title(f'{fail_col}', fontsize=9)
        for j, v in enumerate(fail_corrs.values):
            axes[1, i].text(v + (0.05 if v > 0 else -0.15), j, f'{v:.2f}', va='center', fontsize=7)

plt.suptitle(f'Корреляция < 99.8% (сдвиг погоды ВНУТРИ вышки)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_per_tower_shift.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
if fail_col in tech_cols:
    fig, ax = plt.subplots(figsize=(12, 6))
    for wc in weather_cols:
        values = [correlations[label][fail_col][wc] for label in correlations.keys()]
        ax.plot(shifts, values, marker='o', label=wc, linewidth=2)
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_xlabel('Сдвиг (дни)', fontsize=12)
    ax.set_ylabel('Коэффициент корреляции', fontsize=12)
    ax.set_title(f'Динамика корреляции {fail_col} (сдвиг внутри вышки)', fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(shifts)
    plt.tight_layout()
    plt.savefig('correlation_dynamics_per_tower.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if fail_col in tech_cols:
    print(f"\nТаблица корреляций для {fail_col}:")
    header = f"{'Параметр':<15}" + "".join([f"{'D+{s}':>10}" for s in shifts])
    print(header)
    print("-" * (15 + 10 * len(shifts)))
    for wc in weather_cols:
        row = f"{wc:<15}"
        for s in shifts:
            key = 'No Shift' if s == 0 else f'Shift +{s} Day' if s == 1 else f'Shift +{s} Days'
            c = correlations[key][fail_col][wc] if key in correlations else np.nan
            row += f"{c:10.3f}"
        print(row)

In [ ]:
corr_matrix = X_train.corr(numeric_only=True)

print("\n Топ-20 признаков с наибольшей дисперсией:")
feature_vars = X_train.var(numeric_only=True).sort_values(ascending=False).head(20)
print(feature_vars)

top_features = feature_vars.index.tolist()
corr_top = corr_matrix.loc[top_features, top_features]

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, mask=mask, cmap='coolwarm', vmin=-1, vmax=1,
            center=0, annot=True, fmt='.2f', square=True,
            linewidths=.5, cbar_kws={"shrink": .8})
plt.title('Матрица корреляций (Топ-20 признаков по дисперсии)', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

X_train_with_target = X_train.copy()
X_train_with_target['target'] = np.asarray(y_train).flatten()

corr_matrix_full = X_train_with_target.corr(numeric_only=True)

if 'target' not in corr_matrix_full.columns:
    raise ValueError(" 'target' не попал в корреляционную матрицу. Убедись, что y_train содержит 0 и 1 (int/float).")

corr_with_target = corr_matrix_full['target'].drop('target').sort_values(ascending=False)

print("\n Топ-15 признаков по корреляции с target:")
print(corr_with_target.head(15))

# Визуализация корреляции с target
plt.figure(figsize=(10, 8))
top_corr_features = corr_with_target.head(20).index
corr_values = corr_with_target.loc[top_corr_features]

colors = ['red' if x < 0 else 'green' for x in corr_values.values]
plt.barh(range(len(corr_values)), corr_values.values, color=colors, alpha=0.7)
plt.yticks(range(len(corr_values)), corr_values.index)
plt.xlabel('Корреляция с target')
plt.title('Корреляция признаков с целевой переменной', fontsize=14)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Поиск сильно коррелирующих пар
high_corr_pairs = []
cols = corr_matrix.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.8:
            high_corr_pairs.append({
                'Feature_1': cols[i],
                'Feature_2': cols[j],
                'Correlation': corr_val
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    print("\n СИЛЬНО КОРРЕЛИРУЮЩИЕ ПАРЫ (|corr| > 0.8):")
    print(high_corr_df.to_string(index=False))
else:
    print("\n Сильно коррелирующих пар не найдено.")

# Корреляция по группам
groups = {
    'Weather': [c for c in weather_cols if c in X_train.columns],
    'Weather_Next': [c for c in weather_next_cols if c in X_train.columns],
    'Cyclical': [c for c in cyclical_cols if c in X_train.columns],
    'Interaction': [c for c in interaction_cols if c in X_train.columns],
    'History': [c for c in history_cols if c in X_train.columns],
    'Tech': [c for c in tech_cols if c in X_train.columns]
}

print("\n Средняя корреляция внутри групп:")
for group_name, features in groups.items():
    if len(features) > 1:
        group_corr = corr_matrix.loc[features, features]
        upper_tri = group_corr.where(np.triu(np.ones(group_corr.shape), k=1).astype(bool))
        mean_corr = upper_tri.stack().mean()
        print(f"{group_name:15s}: {mean_corr:.3f}")

# Heatmap исторических признаков
if len(history_cols) > 1:
    hist_features = [c for c in history_cols if c in X_train.columns]
    if len(hist_features) > 1:
        plt.figure(figsize=(10, 8))
        corr_hist = corr_matrix.loc[hist_features, hist_features]
        mask = np.triu(np.ones_like(corr_hist, dtype=bool))
        sns.heatmap(corr_hist, mask=mask, cmap='YlGnBu', annot=True, fmt='.3f',
                    square=True, linewidths=.5)
        plt.title('Корреляция исторических признаков', fontsize=14, pad=20)
        plt.tight_layout()
        plt.show()